# NB-03_proceso_ejec_gc_dipres

Process Flow SAS: **Proceso Ejec GC DIPRES** — `ProcessFlowContainer-FYwV5olig1EcGaWj`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Analista = None  # &Analista — nadie declaró su valor
Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Analista": Analista, "Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=PYDCNIGOB;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-03_proceso_ejec_gc_dipres.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 2a-Check Nvas Ent DIP ACTIV_MOD

Detecta entidades nuevas del año sin clasificación de actividad-nivel en DIPRES, intenta heredar la clasificación del año anterior e imputa por diccionario de partidas/capítulos las que siguen sin clasificar

*confianza: medium · verificador: approve · SAS: PROC SQL: LEFT JOIN de control contra tabla histórica + UPDATE con CASE/diccionario de imputación*

In [ ]:
# ========= 2a-Check Nvas Ent DIP ACTIV_MOD =========
# REVISA SI HAY ENTIDADES NUEVAS SIN ACTIVIDAD CLASIFICADA, EN LA CARGA
# (el bloque comentado del SAS que reconstruía CR18_ENTID_DIP_Y no se traduce: queda como dependencia ya existente en GOBGENER.CR18_ENTID_DIP_Y)
work_conn.execute(text("DROP TABLE IF EXISTS #check_activ_entid_dip"))
# MODIFICADO JUNIO 2025 — VERIFICA ENTIDADES NUEVAS SIN CLASIFICACION ACTIVIDAD-NIVEL
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_check_activ_entid_dip = f"""
SELECT DISTINCT T1.AÑO, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t2.ACTIVIDAD, t2.NIVEL
INTO #check_activ_entid_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y t2
    ON (T1.AÑO = T2.AÑO AND t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA)
WHERE t2.ACTIVIDAD IS NULL AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_check_activ_entid_dip))
check_activ_entid_dip = pd.read_sql(text("SELECT * FROM #check_activ_entid_dip"), work_conn)
_log("check_activ_entid_dip", check_activ_entid_dip)


In [ ]:
# BUSCA ACTIVIDAD-NIVEL EN AÑO ANTERIOR
work_conn.execute(text("DROP TABLE IF EXISTS #check_activ_entid_dip_2"))
# NULL de C_ENTIDAD (LEFT JOIN sin match) se trata explícitamente, no como '<> .' literal
sql_check_activ_entid_dip_2 = """
SELECT DISTINCT T1.AÑO, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t2.ACTIVIDAD, t2.NIVEL, t2.C_ENTIDAD,
       CASE WHEN t2.C_ENTIDAD IS NOT NULL THEN 'Y_1' ELSE '' END AS OBS
INTO #check_activ_entid_dip_2
FROM #check_activ_entid_dip t1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y t2
    ON (T1.AÑO = T2.AÑO + 1 AND t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA)
"""
work_conn.execute(text(sql_check_activ_entid_dip_2))


In [ ]:
# IMPUTAR EN TABLA ACTIVIDAD DE ACUERDO AL SIGUIENTE DICCIONARIO
# PARTIDAS QUE SE CLASIFICAN COMPLETAMENTE EN AAPP-CENTRAL
res = work_conn.execute(text("""
UPDATE #check_activ_entid_dip_2
SET ACTIVIDAD = 'AAPP', NIVEL = 'CENTRAL'
WHERE C_PARTIDA IN ('01','02','03','04','06','07','10','12','13','14','17','18','19','20','21','22','23','24','25','26','27','28','29')
"""))
_log("UPDATE #check_activ_entid_dip_2 (AAPP-CENTRAL partidas)", res.rowcount)


In [ ]:
# EXCEPTO CUANDO CAPITULO ES CASA DE MONEDA
res = work_conn.execute(text("""
UPDATE #check_activ_entid_dip_2
SET ACTIVIDAD = 'AAPP', NIVEL = 'CENTRAL'
WHERE C_PARTIDA = '08' AND C_CAPITULO NOT IN ('06') AND (ACTIVIDAD = '' OR ACTIVIDAD IS NULL)
"""))
_log("UPDATE #check_activ_entid_dip_2 (partida 08 excepto casa de moneda)", res.rowcount)


In [ ]:
# MINISTERIO DEL TRABAJO, CREDITO PRENDARIO
res = work_conn.execute(text("""
UPDATE #check_activ_entid_dip_2
SET ACTIVIDAD = 'EXCLUIDO', NIVEL = 'EXCLUIDO'
WHERE C_PARTIDA = '15' AND C_CAPITULO = '04' AND (ACTIVIDAD = '' OR ACTIVIDAD IS NULL)
"""))
_log("UPDATE #check_activ_entid_dip_2 (partida 15 capitulo 04)", res.rowcount)


In [ ]:
# MINISTERIO DE EDUCACION, CAPITULO 11 JUNJI, SERV LOCALES DE EDUCACION: CAPITULO 18-19-21-22-23-24-25-26-27-28-29-30-31-32-33-34-35-36-37-38-39-40-41-42-43-44-45 AL 56
res = work_conn.execute(text("""
UPDATE #check_activ_entid_dip_2
SET ACTIVIDAD = 'EEPP', NIVEL = 'CENTRAL'
WHERE (ACTIVIDAD = '' OR ACTIVIDAD IS NULL) AND C_PARTIDA = '09' AND
      C_CAPITULO IN ('11','18','19','21','22','23','24','25','26','27','28','29','30','31','32','33','34','35','36','37','38','39','40','41','42','43','44','45','46','47','48','49','50','51','52','53','54','55','56')
"""))
_log("UPDATE #check_activ_entid_dip_2 (partida 09 EEPP)", res.rowcount)


In [ ]:
# Materializa el resultado final de la sesión como DataFrame del notebook (nombre WORK del SAS en minúsculas)
check_activ_entid_dip_2 = pd.read_sql(text("SELECT * FROM #check_activ_entid_dip_2"), work_conn)
_log("check_activ_entid_dip_2", check_activ_entid_dip_2)
# El SAS dropeaba CHECK_ACTIV_ENTID_DIP al final; se limpia la #tmp ya consumida
work_conn.execute(text("DROP TABLE IF EXISTS #check_activ_entid_dip"))


## 2b-Clasif ACTIV de Nvas Ent DIP_MOD

Marca como pendientes de revisión ('REV') las observaciones sin clasificar y anexa las nuevas entidades clasificadas a la matriz maestra de actividades por nivel

*confianza: medium · verificador: approve · SAS: PROC SQL UPDATE sobre WORK + PROC DATASETS APPEND FORCE*

In [ ]:
# ========= 2b-Clasif ACTIV de Nvas Ent DIP_MOD =========
# ACTUALIZAR MATRIZ DE ACTIVIDADES CON IMPUTACIONES PARA NUEVAS ENTIDADES
# MODIFICADO JUNIO 2025
work_conn.execute(text("UPDATE #check_activ_entid_dip_2 SET OBS = 'REV' WHERE OBS IS NULL OR OBS = ''"))


In [ ]:
# PROC DATASETS APPEND FORCE: alinea por nombre de columna, descarta columnas de apoyo que la tabla base no tiene
cols_cr18_entid_dip_y = ", ".join(check_activ_entid_dip_2.columns)
sql_append_cr18_entid_dip_y = f"""
INSERT INTO GOBGENER.dbo.CR18_ENTID_DIP_Y ({cols_cr18_entid_dip_y})
SELECT {cols_cr18_entid_dip_y}
FROM #check_activ_entid_dip_2
"""
res = work_conn.execute(text(sql_append_cr18_entid_dip_y))
_log("APPEND GOBGENER.dbo.CR18_ENTID_DIP_Y", res.rowcount)


## 1a-Check Analiticos Predef DIP_MOD

Identifica analíticos fijos (cuentas distintas de 05/13/24/33) sin clasificación en la tabla maestro CR18_T_SCN_1 y propone una clasificación reutilizando el historial más frecuente por clave CUENTA-ITEM-ASIGNACION (completa y abreviada)

*confianza: medium · verificador: revise · SAS: PROC SQL (WORK como #tmp de sesión) con LEFT JOIN, GROUP BY, PROC SORT NODUPKEY y UNION ALL final*

In [ ]:
# ========= 1a-Check Analiticos Predef DIP_MOD =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_ejec_dip = f"""
SELECT *
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES
WHERE AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_dip))


In [ ]:
# DETERMINA ANALITICOS QUE NO ESTÁN EN TABLA FIJA Y QUE REQUIEREN CLASIFICACIÓN
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_faltante_d"))
sql_analit_fijo_faltante_d = """
SELECT t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       COUNT(t1.DEVENG) AS ID, t2.C_ANALITICO AS C_ANALIT,
       CONCAT_WS('_', t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION) AS KEY_CORTA,
       CONCAT_WS('_', t1.C_CUENTA, t1.C_ITEM, SUBSTRING(t1.C_ASIGNACION, 1, 2)) AS KEY_CORTA_2
INTO #analit_fijo_faltante_d
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON (t1.C_TIPO_CUENTA = t2.T_CUENTA) AND (t1.C_CUENTA = t2.C_CUENTA) AND (t1.C_ANALITICO = t2.C_ANALITICO)
WHERE (t2.C_ANALITICO IS NULL OR t2.C_ANALITICO = '') AND t1.C_CUENTA NOT IN ('05','13','24','33')
GROUP BY t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t2.C_ANALITICO
"""
work_conn.execute(text(sql_analit_fijo_faltante_d))
analit_fijo_faltante_d = pd.read_sql(text("SELECT * FROM #analit_fijo_faltante_d"), work_conn)
_log("analit_fijo_faltante_d", analit_fijo_faltante_d)


In [ ]:
# II. PROPONE CLASIFICACIONES A LOS FALTANTES DEL PERIODO, DEFINIENDO CLAVE Y USANDO LAS CLASIFICACIONES QUE TIENEN MAYOR FRECUENCIA
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO CUENTA-ITEM-ASIGNACION
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo"))
sql_cruce_fijo = """
SELECT CONCAT_WS('_', C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_CORTA,
       T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION,
       N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_fijo
FROM GOBGENER.dbo.CR18_T_SCN_1
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, T_TRANSACCION
"""
work_conn.execute(text(sql_cruce_fijo))


In [ ]:
# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY_CORTA
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo_max"))
sql_cruce_fijo_max = """
SELECT KEY_CORTA, MAX(ID) AS ID_MAX
INTO #cruce_fijo_max
FROM #cruce_fijo
GROUP BY KEY_CORTA
"""
work_conn.execute(text(sql_cruce_fijo_max))


In [ ]:
# CLASIFICACION PROPUESTA ANALITICOS FIJOS
# Se agrega ORDER BY (SELECT NULL) para preservar el orden físico de inserción de #cruce_fijo
# (SAS no tenía ORDER BY explícito; SQL Server no garantiza orden sin ORDER BY, así que
# se emula el orden de creación con una columna identity de apoyo antes del NODUPKEY)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo_ord"))
sql_cruce_fijo_ord = """
SELECT *, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS __ORDEN__
INTO #cruce_fijo_ord
FROM #cruce_fijo
"""
work_conn.execute(text(sql_cruce_fijo_ord))

work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_all"))
sql_cruce_prop_all = """
SELECT T1.*
INTO #cruce_prop_all
FROM #cruce_fijo_ord T1, #cruce_fijo_max T2
WHERE T1.KEY_CORTA = T2.KEY_CORTA AND T1.ID = T2.ID_MAX
"""
work_conn.execute(text(sql_cruce_prop_all))


In [ ]:
# PROC SORT DATA=CRUCE_PROP OUT=CRUCE_PROP NODUPKEY; BY KEY_CORTA;
# ELIMINA POSIBLES DUPLICADOS conservando, ante empate de KEY_CORTA/ID_MAX, la fila que
# apareció primero en el orden físico de creación de #cruce_fijo (columna __ORDEN__),
# no un orden alfabético de KEY_CORTA (que cambiaría el desempate frente al SAS original)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop"))
sql_cruce_prop = """
SELECT KEY_CORTA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION,
       N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ID
INTO #cruce_prop
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY KEY_CORTA ORDER BY __ORDEN__) AS __RN__
    FROM #cruce_prop_all
) sub
WHERE __RN__ = 1
"""
work_conn.execute(text(sql_cruce_prop))
cruce_prop = pd.read_sql(text("SELECT * FROM #cruce_prop"), work_conn)
_log("cruce_prop", cruce_prop)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, CON UNA CLAVE MAS CORTA
# QUE LA ANTERIOR (CUENTA-ITEM-ASIGNACION, CON ASIGNACION USANDO LOS DOS PRIMEROS DIGITOS)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo_2"))
sql_cruce_fijo_2 = """
SELECT CONCAT_WS('_', C_CUENTA, C_ITEM, SUBSTRING(C_ASIGNACION, 1, 2)) AS KEY_CORTA_2,
       C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_fijo_2
FROM GOBGENER.dbo.CR18_T_SCN_1
GROUP BY C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO,
         CONCAT_WS('_', C_CUENTA, C_ITEM, SUBSTRING(C_ASIGNACION, 1, 2))
"""
work_conn.execute(text(sql_cruce_fijo_2))


In [ ]:
# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY_CORTA_2
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo_max_2"))
sql_cruce_fijo_max_2 = """
SELECT KEY_CORTA_2, MAX(ID) AS ID_MAX
INTO #cruce_fijo_max_2
FROM #cruce_fijo_2
GROUP BY KEY_CORTA_2
"""
work_conn.execute(text(sql_cruce_fijo_max_2))


In [ ]:
# CLASIFICACION PROPUESTA ANALITICOS FIJOS (clave corta 2)
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_fijo_2_ord"))
sql_cruce_fijo_2_ord = """
SELECT *, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS __ORDEN__
INTO #cruce_fijo_2_ord
FROM #cruce_fijo_2
"""
work_conn.execute(text(sql_cruce_fijo_2_ord))

work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_2_all"))
sql_cruce_prop_2_all = """
SELECT T1.*
INTO #cruce_prop_2_all
FROM #cruce_fijo_2_ord T1, #cruce_fijo_max_2 T2
WHERE T1.KEY_CORTA_2 = T2.KEY_CORTA_2 AND T1.ID = T2.ID_MAX
"""
work_conn.execute(text(sql_cruce_prop_2_all))


In [ ]:
# PROC SORT DATA=CRUCE_PROP_2 OUT=CRUCE_PROP_2 NODUPKEY; BY KEY_CORTA_2;
# mismo criterio de desempate que CRUCE_PROP: primera fila según el orden físico
# de creación de #cruce_fijo_2 (__ORDEN__), no orden alfabético de KEY_CORTA_2
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_2"))
sql_cruce_prop_2 = """
SELECT KEY_CORTA_2, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ID
INTO #cruce_prop_2
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY KEY_CORTA_2 ORDER BY __ORDEN__) AS __RN__
    FROM #cruce_prop_2_all
) sub
WHERE __RN__ = 1
"""
work_conn.execute(text(sql_cruce_prop_2))
cruce_prop_2 = pd.read_sql(text("SELECT * FROM #cruce_prop_2"), work_conn)
_log("cruce_prop_2", cruce_prop_2)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES EN PREDEFINIDOS, USANDO CLAVE 1 (CUENTA-ITEM-ASIGNACION)
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_clas"))
sql_analit_fijo_clas = f"""
SELECT t1.C_TIPO_CUENTA AS T_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO AS ANALITICO, t2.C_SCN, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_CAGENTE, t2.C_CUP, t2.C_ACTIVO,
       CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA,
       t1.KEY_CORTA, t1.KEY_CORTA_2
INTO #analit_fijo_clas
FROM #analit_fijo_faltante_d t1
LEFT JOIN #cruce_prop t2 ON (t1.KEY_CORTA = t2.KEY_CORTA)
"""
work_conn.execute(text(sql_analit_fijo_clas))
analit_fijo_clas = pd.read_sql(text("SELECT * FROM #analit_fijo_clas"), work_conn)
_log("analit_fijo_clas", analit_fijo_clas)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES EN PREDEFINIDOS, USANDO CLAVE CORTA 2
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_clas_2"))
sql_analit_fijo_clas_2 = f"""
SELECT t1.T_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.ANALITICO,
       t2.C_SCN, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_CAGENTE, t2.C_CUP, t2.C_ACTIVO,
       CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA
INTO #analit_fijo_clas_2
FROM #analit_fijo_clas t1
LEFT JOIN #cruce_prop_2 t2 ON (t1.KEY_CORTA_2 = t2.KEY_CORTA_2)
WHERE (t1.C_SCN IS NULL OR t1.C_SCN = '')
"""
work_conn.execute(text(sql_analit_fijo_clas_2))
analit_fijo_clas_2 = pd.read_sql(text("SELECT * FROM #analit_fijo_clas_2"), work_conn)
_log("analit_fijo_clas_2", analit_fijo_clas_2)


In [ ]:
# tabla con clasificaciones propuestas. Revisar y modificar en caso que sea necesario
# DATA ANALIT_FIJO_CLAS_D; SET ANALIT_FIJO_CLAS ANALIT_FIJO_CLAS_2; WHERE C_SCN<>''; DROP KEY_CORTA KEY_CORTA_2;
# ambas fuentes comparten columnas salvo KEY_CORTA/KEY_CORTA_2 (propias de #analit_fijo_clas);
# se listan explícitas para el UNION ALL y se excluyen esas dos
cols_clas_d = "T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, FECHA, ANALISTA"
work_conn.execute(text("DROP TABLE IF EXISTS #analit_fijo_clas_d"))
sql_analit_fijo_clas_d = f"""
SELECT {cols_clas_d}
INTO #analit_fijo_clas_d
FROM (
    SELECT {cols_clas_d} FROM #analit_fijo_clas WHERE C_SCN <> ''
    UNION ALL
    SELECT {cols_clas_d} FROM #analit_fijo_clas_2 WHERE C_SCN <> ''
) u
"""
work_conn.execute(text(sql_analit_fijo_clas_d))
analit_fijo_clas_d = pd.read_sql(text("SELECT * FROM #analit_fijo_clas_d"), work_conn)
_log("analit_fijo_clas_d", analit_fijo_clas_d)


In [ ]:
# limpieza de tablas temporales de sesión (equivalente al DROP TABLE final del SAS)
for t in ["#cruce_fijo", "#cruce_fijo_max", "#cruce_fijo_2", "#cruce_fijo_max_2",
          "#analit_fijo_clas_2", "#cruce_prop", "#cruce_prop_2", "#ejec_dip",
          "#cruce_fijo_ord", "#cruce_prop_all", "#cruce_fijo_2_ord", "#cruce_prop_2_all"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## 1b-Clasif Analiticos Predef DIP_MOD

Anexa a la tabla maestra de clasificación de analíticos las clasificaciones nuevas del paso previo (evitando duplicados vía borrado previo), marca como pendientes de revisión ('REV') los registros sin clasificación asignada, y deja tablas de control de duplicados y pendientes

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE EXISTS + UPDATE + PROC DATASETS APPEND FORCE + CREATE TABLE de verificación*

In [ ]:
# ========= 1b-Clasif Analiticos Predef DIP_MOD =========
# I. ANEXA CLASIFICACIONES FALTANTES DESDE PASO ANTERIOR
# ANALIT_FIJO_CLAS_D es una #tmp de sesión dejada por el nodo anterior (1a)
work_conn.execute(text("""
DELETE FROM GOBGENER.dbo.CR18_T_SCN_1
WHERE EXISTS (
    SELECT 1 FROM #analit_fijo_clas_d
    WHERE #analit_fijo_clas_d.T_CUENTA = CR18_T_SCN_1.T_CUENTA
      AND #analit_fijo_clas_d.C_CUENTA = CR18_T_SCN_1.C_CUENTA
      AND #analit_fijo_clas_d.C_ITEM = CR18_T_SCN_1.C_ITEM
      AND #analit_fijo_clas_d.C_ASIGNACION = CR18_T_SCN_1.C_ASIGNACION
      AND #analit_fijo_clas_d.C_ANALITICO = CR18_T_SCN_1.C_ANALITICO
)
"""))


In [ ]:
# Registros pendientes de clasificación (C_SCN vacío o NULL) quedan marcados como 'REV'
res = work_conn.execute(text("""
UPDATE #analit_fijo_clas_d
SET C_SCN = 'REV'
WHERE C_SCN IS NULL OR C_SCN = ''
"""))
_log("UPDATE #analit_fijo_clas_d C_SCN='REV'", res.rowcount)


In [ ]:
# PROC APPEND FORCE: alinea por nombre de columna, server-side desde la #tmp
cols_cr18 = "T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, C_SCN"
sql_append = f"""
INSERT INTO GOBGENER.dbo.CR18_T_SCN_1 ({cols_cr18})
SELECT {cols_cr18}
FROM #analit_fijo_clas_d
"""
res = work_conn.execute(text(sql_append))
_log("APPEND GOBGENER.dbo.CR18_T_SCN_1", res.rowcount)


In [ ]:
# VERIFICA QUE NO EXISTAN DUPLICADOS EN LA BASE DE CLASIFICACIONES FIJAS
work_conn.execute(text("DROP TABLE IF EXISTS #check_analit_fijo"))
sql_check_analit_fijo = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, COUNT(C_SCN) AS ID
INTO #check_analit_fijo
FROM GOBGENER.dbo.CR18_T_SCN_1
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO
"""
work_conn.execute(text(sql_check_analit_fijo))
# Si hay repetidos mostrará campos en esta salida: se conservan solo grupos con ID > 1
work_conn.execute(text("DELETE FROM #check_analit_fijo WHERE ID = 1"))
check_analit_fijo = pd.read_sql(text("SELECT * FROM #check_analit_fijo"), work_conn)
_log("check_analit_fijo", check_analit_fijo)


In [ ]:
# VERIFICA QUE NO EXISTAN ANALITICOS SIN CLASIFICAR (marcados 'REV')
work_conn.execute(text("DROP TABLE IF EXISTS #check_analit_rev"))
sql_check_analit_rev = """
SELECT T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, C_SCN
INTO #check_analit_rev
FROM GOBGENER.dbo.CR18_T_SCN_1
WHERE C_SCN = 'REV'
"""
work_conn.execute(text(sql_check_analit_rev))
check_analit_rev = pd.read_sql(text("SELECT * FROM #check_analit_rev"), work_conn)
_log("check_analit_rev", check_analit_rev)


## OPC 1a-Check Analiticos Var DIP_MOD

Detecta analíticos variables (cuentas 05,13,24,33) sin clasificación SCN vigente y propone la clasificación más frecuente por entidad-partida-programa, en cascada de 5 claves (GORE primero, luego resto de entidades, luego por entidad sola), dejando un check para revisión manual antes de exportar a Excel

*confianza: medium · verificador: approve · SAS: PROC SQL en cascada (5 niveles de claves compositas) con LEFT JOINs, NODUPKEY, UNION y UPDATE sobre tablas temporales de sesión*

In [ ]:
# ========= OPC 1a-Check Analiticos Var DIP_MOD =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
# MODIFICADO EN JUNIO 2025 / Ejecucion dipres del año DIPRES
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_dip))


In [ ]:
# I. DETERMINA ANALITICOS FALTANTES VARIABLES POR CLASIFICAR DEL PERIODO
work_conn.execute(text("DROP TABLE IF EXISTS #analit_entidad_var_d"))
# ANALITICOS QUE NO ESTÁN EN TABLA DE CLASIFICACIONES
# T2.C_ANALITICO="" tras LEFT JOIN: en SQL Server la fila sin match queda NULL, no ''
sql_analit_entidad_var_d = """
SELECT T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA, T1.C_CUENTA,
       T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, COUNT(T1.DEVENG) AS ID,
       T2.C_ANALITICO AS C_ANALIT
INTO #analit_entidad_var_d
FROM #ejec_dip T1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 T2
    ON T1.C_ENTIDAD = T2.ENTIDAD AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO
   AND T1.C_PROGRAMA = T2.C_PROGRAMA AND T1.C_TIPO_CUENTA = T2.T_CUENTA AND T1.C_CUENTA = T2.C_CUENTA
   AND T1.C_ANALITICO = T2.C_ANALITICO
WHERE (T2.C_ANALITICO IS NULL OR T2.C_ANALITICO = '') AND T1.C_CUENTA IN ('05','13','24','33')
GROUP BY T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA, T1.C_CUENTA,
         T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, T2.C_ANALITICO
"""
work_conn.execute(text(sql_analit_entidad_var_d))


In [ ]:
# II. PROPONE CLASIFICACIONES PARA GORE (PARTIDA=88)
work_conn.execute(text("""
ALTER TABLE #analit_entidad_var_d ADD KEY_1 varchar(200), KEY_2 varchar(200)
"""))
work_conn.execute(text("""
UPDATE #analit_entidad_var_d
SET KEY_1 = CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION),
    KEY_2 = CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ASIGNACION)
"""))
analit_entidad_var_d = pd.read_sql(text("SELECT * FROM #analit_entidad_var_d"), work_conn)
_log("analit_entidad_var_d", analit_entidad_var_d)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_1
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var"))
sql_cruce_var = """
SELECT CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_1,
       T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA,
       C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA = '88'
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP,
         C_ACTIVO, T_TRANSACCION, CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION)
"""
work_conn.execute(text(sql_cruce_var))

# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY_CORTA
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max"))
work_conn.execute(text("""
SELECT KEY_1, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max
FROM #cruce_var
GROUP BY KEY_1
"""))

# CLASIFICACION PROPUESTA ANALITICOS FIJOS
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop"))
work_conn.execute(text("""
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop
FROM #cruce_var T1, #cruce_var_max T2
WHERE T1.KEY_1 = T2.KEY_1 AND T1.ID = T2.ID_MAX
"""))

# ELIMINA POSIBLES DUPLICADOS (NODUPKEY BY KEY_1) -- se materializa vía pandas por simplicidad del sort/dedup
cruce_prop = pd.read_sql(text("SELECT * FROM #cruce_prop ORDER BY KEY_1"), work_conn)
cruce_prop = cruce_prop.drop_duplicates(subset=["KEY_1"], keep="first")
cruce_prop.to_sql("#cruce_prop", work_conn, if_exists="replace", index=False)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_2
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_2"))
sql_cruce_var_2 = """
SELECT CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ASIGNACION) AS KEY_2,
       T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA,
       C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var_2
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA = '88'
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP,
         C_ACTIVO, T_TRANSACCION, CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ASIGNACION)
"""
work_conn.execute(text(sql_cruce_var_2))

# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY_CORTA
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max_2"))
work_conn.execute(text("""
SELECT KEY_2, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max_2
FROM #cruce_var_2
GROUP BY KEY_2
"""))

# CLASIFICACION PROPUESTA ANALITICOS FIJOS, KEY_2
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_2"))
work_conn.execute(text("""
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop_2
FROM #cruce_var_2 T1, #cruce_var_max_2 T2
WHERE T1.KEY_2 = T2.KEY_2 AND T1.ID = T2.ID_MAX
"""))

# ELIMINA POSIBLES DUPLICADOS (NODUPKEY BY KEY_2)
cruce_prop_2 = pd.read_sql(text("SELECT * FROM #cruce_prop_2 ORDER BY KEY_2"), work_conn)
cruce_prop_2 = cruce_prop_2.drop_duplicates(subset=["KEY_2"], keep="first")
cruce_prop_2.to_sql("#cruce_prop_2", work_conn, if_exists="replace", index=False)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES GORES, USANDO KEY_1
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas"))
# año/analista interpolados como literales controlados (no :param) para que la #tmp sobreviva
sql_analit_var_clas = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA AS T_CUENTA, T1.C_CUENTA,
       T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO AS ANALITICO, T2.C_SCN, T2.C_TRANSACCION,
       T2.T_TRANSACCION, T2.N_CUENTA, T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.C_ENTIDAD AS ENTIDAD,
       CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA, T1.KEY_1, T1.KEY_2, T2.ID_COUNT
INTO #analit_var_clas
FROM #analit_entidad_var_d T1
LEFT JOIN #cruce_prop T2 ON T1.KEY_1 = T2.KEY_1
WHERE T1.C_PARTIDA = '88'
"""
work_conn.execute(text(sql_analit_var_clas))


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES GORES, USANDO KEY_2
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_2"))
# C_SCN='' de SAS: acá T1 viene de #analit_var_clas (creado arriba, C_SCN puede ser NULL si no matcheó CRUCE_PROP)
sql_analit_var_clas_2 = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.T_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION,
       T1.C_ANALITICO, T1.ANALITICO, T2.C_SCN, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA,
       T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.ENTIDAD, CAST(GETDATE() AS date) AS FECHA,
       UPPER('{Analista}') AS ANALISTA, T2.ID_COUNT
INTO #analit_var_clas_2
FROM #analit_var_clas T1
LEFT JOIN #cruce_prop_2 T2 ON T1.KEY_2 = T2.KEY_2
WHERE (T1.C_SCN IS NULL OR T1.C_SCN = '')
"""
work_conn.execute(text(sql_analit_var_clas_2))

# DELETE FROM ANALIT_VAR_CLAS WHERE C_SCN=''
res = work_conn.execute(text("DELETE FROM #analit_var_clas WHERE C_SCN IS NULL OR C_SCN = ''"))
_log("DELETE #analit_var_clas (C_SCN vacio)", res.rowcount)


In [ ]:
# UNE clasificaciones GORE: ANALIT_VAR_CLAS + ANALIT_VAR_CLAS_2, sin KEY_1/KEY_2
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_gore"))
work_conn.execute(text("""
SELECT C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
       ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ENTIDAD,
       FECHA, ANALISTA, ID_COUNT
INTO #analit_var_clas_gore
FROM #analit_var_clas
UNION ALL
SELECT C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
       ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ENTIDAD,
       FECHA, ANALISTA, ID_COUNT
FROM #analit_var_clas_2
"""))
analit_var_clas_gore = pd.read_sql(text("SELECT * FROM #analit_var_clas_gore"), work_conn)
_log("analit_var_clas_gore", analit_var_clas_gore)

# limpieza de temporales intermedias de este bloque
for t in ["#cruce_var", "#cruce_var_max", "#cruce_var_2", "#cruce_var_max_2", "#analit_var_clas_2", "#cruce_prop", "#cruce_prop_2", "#analit_var_clas"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# III. PROPONE CLASIFICACIONES PARA RESTO DE ENTIDADES (DISTINTOS DE GORE)
work_conn.execute(text("ALTER TABLE #analit_entidad_var_d DROP COLUMN KEY_1, KEY_2"))
work_conn.execute(text("ALTER TABLE #analit_entidad_var_d ADD KEY_3 varchar(200), KEY_4 varchar(200)"))
work_conn.execute(text("""
UPDATE #analit_entidad_var_d
SET KEY_3 = CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION),
    KEY_4 = CONCAT_WS('_', C_PARTIDA, C_PROGRAMA, C_CUENTA, C_ASIGNACION)
"""))


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_3
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_3"))
work_conn.execute(text("""
SELECT CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_3,
       T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA,
       C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var_3
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA NOT IN ('88')
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP,
         C_ACTIVO, T_TRANSACCION, CONCAT_WS('_', C_PARTIDA, C_CUENTA, C_ITEM, C_ASIGNACION)
"""))

# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max_3"))
work_conn.execute(text("""
SELECT KEY_3, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max_3
FROM #cruce_var_3
GROUP BY KEY_3
"""))

# CLASIFICACION PROPUESTA ANALITICOS FIJOS, KEY_3
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_3"))
work_conn.execute(text("""
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop_3
FROM #cruce_var_3 T1, #cruce_var_max_3 T2
WHERE T1.KEY_3 = T2.KEY_3 AND T1.ID = T2.ID_MAX
"""))

# ELIMINA POSIBLES DUPLICADOS (NODUPKEY BY KEY_3)
cruce_prop_3 = pd.read_sql(text("SELECT * FROM #cruce_prop_3 ORDER BY KEY_3"), work_conn)
cruce_prop_3 = cruce_prop_3.drop_duplicates(subset=["KEY_3"], keep="first")
cruce_prop_3.to_sql("#cruce_prop_3", work_conn, if_exists="replace", index=False)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_4
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_4"))
work_conn.execute(text("""
SELECT CONCAT_WS('_', C_PARTIDA, C_PROGRAMA, C_CUENTA, C_ASIGNACION) AS KEY_4,
       T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA,
       C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var_4
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA NOT IN ('88')
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP,
         C_ACTIVO, T_TRANSACCION, CONCAT_WS('_', C_PARTIDA, C_PROGRAMA, C_CUENTA, C_ASIGNACION)
"""))

# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max_4"))
work_conn.execute(text("""
SELECT KEY_4, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max_4
FROM #cruce_var_4
GROUP BY KEY_4
"""))

# CLASIFICACION PROPUESTA ANALITICOS FIJOS, KEY_4
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_4"))
work_conn.execute(text("""
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop_4
FROM #cruce_var_4 T1, #cruce_var_max_4 T2
WHERE T1.KEY_4 = T2.KEY_4 AND T1.ID = T2.ID_MAX
"""))

# ELIMINA POSIBLES DUPLICADOS (NODUPKEY BY KEY_4)
cruce_prop_4 = pd.read_sql(text("SELECT * FROM #cruce_prop_4 ORDER BY KEY_4"), work_conn)
cruce_prop_4 = cruce_prop_4.drop_duplicates(subset=["KEY_4"], keep="first")
cruce_prop_4.to_sql("#cruce_prop_4", work_conn, if_exists="replace", index=False)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES, USANDO KEY_3
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_3"))
sql_analit_var_clas_3 = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA AS T_CUENTA, T1.C_CUENTA,
       T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO AS ANALITICO, T2.C_SCN, T2.C_TRANSACCION,
       T2.T_TRANSACCION, T2.N_CUENTA, T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.C_ENTIDAD AS ENTIDAD,
       CAST(GETDATE() AS date) AS FECHA, UPPER('{Analista}') AS ANALISTA, T1.KEY_3, T1.KEY_4, T2.ID_COUNT
INTO #analit_var_clas_3
FROM #analit_entidad_var_d T1
LEFT JOIN #cruce_prop_3 T2 ON T1.KEY_3 = T2.KEY_3
WHERE T1.C_PARTIDA NOT IN ('88')
"""
work_conn.execute(text(sql_analit_var_clas_3))

# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES, USANDO KEY_4
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_4"))
sql_analit_var_clas_4 = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.T_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION,
       T1.C_ANALITICO, T1.ANALITICO, T2.C_SCN, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA,
       T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.ENTIDAD, CAST(GETDATE() AS date) AS FECHA,
       UPPER('{Analista}') AS ANALISTA, T1.KEY_3, T1.KEY_4, T2.ID_COUNT
INTO #analit_var_clas_4
FROM #analit_var_clas_3 T1
LEFT JOIN #cruce_prop_4 T2 ON T1.KEY_4 = T2.KEY_4
WHERE T1.C_PARTIDA NOT IN ('88') AND (T1.C_SCN IS NULL OR T1.C_SCN = '')
"""
work_conn.execute(text(sql_analit_var_clas_4))

# DELETE FROM ANALIT_VAR_CLAS_3 WHERE C_SCN=''
res = work_conn.execute(text("DELETE FROM #analit_var_clas_3 WHERE C_SCN IS NULL OR C_SCN = ''"))
_log("DELETE #analit_var_clas_3 (C_SCN vacio)", res.rowcount)


In [ ]:
# UNE clasificaciones RESTO: ANALIT_VAR_CLAS_3 + ANALIT_VAR_CLAS_4, sin KEY_3/KEY_4
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_resto"))
work_conn.execute(text("""
SELECT C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
       ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ENTIDAD,
       FECHA, ANALISTA, ID_COUNT
INTO #analit_var_clas_resto
FROM #analit_var_clas_3
UNION ALL
SELECT C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO,
       ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ENTIDAD,
       FECHA, ANALISTA, ID_COUNT
FROM #analit_var_clas_4
"""))

# limpieza de temporales intermedias de este bloque
for t in ["#cruce_var_3", "#cruce_var_max_3", "#cruce_var_4", "#cruce_var_max_4", "#analit_var_clas_4", "#cruce_prop_3", "#cruce_prop_4", "#analit_var_clas_3"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# IV. PROPONE CLASIFICACIONES PARA RESTO DE ENTIDADES, FALTANTES DE PASO ANTERIOR, SI SE MANTIENE ENTIDAD PERO CAMBIA PARTIDA
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_resto_f"))
work_conn.execute(text("""
SELECT *, CONCAT_WS('_', ENTIDAD, C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_5
INTO #analit_var_clas_resto_f
FROM #analit_var_clas_resto
WHERE C_SCN IS NULL OR C_SCN = ''
"""))
analit_var_clas_resto_f = pd.read_sql(text("SELECT * FROM #analit_var_clas_resto_f"), work_conn)
_log("analit_var_clas_resto_f", analit_var_clas_resto_f)


In [ ]:
# GENERA CLAVES EN LO YA CLASIFICADO PARA REUTILIZAR Y CLASIFICAR LO NUEVO, USANDO KEY_5
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_5"))
work_conn.execute(text("""
SELECT CONCAT_WS('_', ENTIDAD, C_CUENTA, C_ITEM, C_ASIGNACION) AS KEY_5,
       T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA,
       C_CAGENTE, C_CUP, C_ACTIVO, COUNT(FECHA) AS ID
INTO #cruce_var_5
FROM GOBGENER.dbo.CR18_T_SCN_2
WHERE C_PARTIDA NOT IN ('88')
GROUP BY T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_SCN, C_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP,
         C_ACTIVO, T_TRANSACCION, CONCAT_WS('_', ENTIDAD, C_CUENTA, C_ITEM, C_ASIGNACION)
"""))

# DEFINE LINEA CON MAYOR NUMERO DE REPETICIONES PARA CADA KEY
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_var_max_5"))
work_conn.execute(text("""
SELECT KEY_5, MAX(ID) AS ID_MAX, COUNT(ID) AS ID
INTO #cruce_var_max_5
FROM #cruce_var_5
GROUP BY KEY_5
"""))

# CLASIFICACION PROPUESTA ANALITICOS FIJOS, KEY_5
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_prop_5"))
work_conn.execute(text("""
SELECT T1.*, T2.ID_MAX, T2.ID AS ID_COUNT
INTO #cruce_prop_5
FROM #cruce_var_5 T1, #cruce_var_max_5 T2
WHERE T1.KEY_5 = T2.KEY_5 AND T1.ID = T2.ID_MAX
"""))

# ELIMINA POSIBLES DUPLICADOS (NODUPKEY BY KEY_5)
cruce_prop_5 = pd.read_sql(text("SELECT * FROM #cruce_prop_5 ORDER BY KEY_5"), work_conn)
cruce_prop_5 = cruce_prop_5.drop_duplicates(subset=["KEY_5"], keep="first")
cruce_prop_5.to_sql("#cruce_prop_5", work_conn, if_exists="replace", index=False)


In [ ]:
# ASIGNA CLASIFICACIONES PROPUESTAS A FALTANTES, USANDO KEY_5
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_5"))
sql_analit_var_clas_5 = f"""
SELECT T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.T_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION,
       T1.C_ANALITICO, T1.ANALITICO, T2.C_SCN, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA,
       T2.C_CAGENTE, T2.C_CUP, T2.C_ACTIVO, T1.ENTIDAD, CAST(GETDATE() AS date) AS FECHA,
       UPPER('{Analista}') AS ANALISTA, T2.ID_COUNT
INTO #analit_var_clas_5
FROM #analit_var_clas_resto_f T1
LEFT JOIN #cruce_prop_5 T2 ON T1.KEY_5 = T2.KEY_5
"""
work_conn.execute(text(sql_analit_var_clas_5))
analit_var_clas_5 = pd.read_sql(text("SELECT * FROM #analit_var_clas_5"), work_conn)
_log("analit_var_clas_5", analit_var_clas_5)

# limpieza de temporales intermedias de este bloque
for t in ["#cruce_var_5", "#cruce_var_max_5", "#cruce_prop_5"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# UNE TABLAS FINALES DE CLASIFICACIONES PROPUESTAS
# ANALIT_VAR_CLAS_RESTO se filtra a C_SCN<>'' (los que sí quedaron clasificados)
work_conn.execute(text("DROP TABLE IF EXISTS #analit_var_clas_resto_ok"))
work_conn.execute(text("""
SELECT *
INTO #analit_var_clas_resto_ok
FROM #analit_var_clas_resto
WHERE C_SCN IS NOT NULL AND C_SCN <> ''
"""))

work_conn.execute(text("DROP TABLE IF EXISTS #check_transvar"))
cols_union = "C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ANALITICO, C_SCN, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_CAGENTE, C_CUP, C_ACTIVO, ENTIDAD, FECHA, ANALISTA"
work_conn.execute(text(f"""
SELECT {cols_union}, 'Ingresar texto si aplica' AS OBS
INTO #check_transvar
FROM #analit_var_clas_gore
UNION ALL
SELECT {cols_union}, 'Ingresar texto si aplica' AS OBS
FROM #analit_var_clas_resto_ok
UNION ALL
SELECT {cols_union}, 'Ingresar texto si aplica' AS OBS
FROM #analit_var_clas_5
"""))


In [ ]:
# actualizaciones de OBS/C_ACTIVO según C_SCN/C_CUP
res = work_conn.execute(text("UPDATE #check_transvar SET OBS = 'CR18_difcoy', C_ACTIVO = 'D63' WHERE C_SCN IN ('D63-P2','D63-P')"))
_log("UPDATE #check_transvar D63", res.rowcount)
res = work_conn.execute(text("UPDATE #check_transvar SET OBS = 'CR18_difact' WHERE C_SCN = 'P3'"))
_log("UPDATE #check_transvar P3", res.rowcount)
res = work_conn.execute(text("UPDATE #check_transvar SET OBS = 'CR18_difact' WHERE C_SCN = 'P2' AND C_CUP IN ('EEPP','SSPP')"))
_log("UPDATE #check_transvar P2", res.rowcount)

check_transvar = pd.read_sql(text("SELECT * FROM #check_transvar"), work_conn)
_log("check_transvar", check_transvar)

# limpieza de temporales de este nodo (equivalente a DROP TABLE ANALIT_VAR_CLAS_GORE, ANALIT_VAR_CLAS_RESTO, ANALIT_VAR_CLAS_5, ANALIT_VAR_CLAS_RESTO_F, EJEC_DIP)
for t in ["#analit_var_clas_gore", "#analit_var_clas_resto", "#analit_var_clas_resto_ok", "#analit_var_clas_5", "#analit_var_clas_resto_f", "#ejec_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# PROC EXPORT a Excel binario (.xlsb) vía bridge PC Files Server: se traduce como escritura directa por ruta relativa
# ruta original UNC del servidor de archivos reemplazada por ruta relativa al workspace
salida_check_transvar = Path("salidas") / "CHECK_TRANSVAR_PROP_D.xlsb"
# pandas/openpyxl no pueden escribir formato .xlsb; se exporta a .xlsx equivalente manteniendo la hoja
salida_check_transvar_xlsx = salida_check_transvar.with_suffix(".xlsx")
check_transvar.to_excel(salida_check_transvar_xlsx, sheet_name="PROPUESTA", index=False)
_log("export check_transvar", salida_check_transvar_xlsx)


## OPC 1b-Clasif Analiticos Var DIP_MOD

Importa la propuesta de clasificación analítica variable revisada por el analista, marca diferencias de concepto (CR18_difcoy) y de actividad (CR18_difact) según reglas de C_SCN/C_CUP, anexa a la tabla maestra CR18_T_SCN_2 solo las combinaciones nuevas, y deja un control de duplicados por clave

*confianza: medium · verificador: approve · SAS: PROC IMPORT (bridge EXCELCS) + PROC SQL UPDATE/DELETE de reglas de negocio + anti-join anexo (PROC APPEND FORCE) + verificación de duplicados*

In [ ]:
# ========= OPC 1b-Clasif Analiticos Var DIP_MOD =========
# MODIFICADO JUNIO 2025
# IMPORTA CLASIFICACIONES
# Ruta relativa al workspace: el SAS original leía desde una ruta UNC de red
# (\\\\FILESERVER\\VOLM\\GEM\\050_DCNI\\02_CNSI\\16_GOBIERNO\\GobiernoGeneral\\CHECK_TRANSVAR_PROP_D.xlsb)
# vía el bridge PC Files Server (DBMS=EXCELCS + SERVER_NAME/SERVERUSER/SERVERPASS);
# en Python se lee el archivo directo, sin servidor intermedio ni credenciales.
ruta_check_transvar_p = Path("insumos") / "CHECK_TRANSVAR_PROP_D.xlsb"
check_transvar_p = pd.read_excel(ruta_check_transvar_p, sheet_name="PROPUESTA", engine="pyxlsb")
_log("check_transvar_p", check_transvar_p)


In [ ]:
# Sube el insumo importado a una tabla temporal de sesión para operar el resto del nodo en SQL
work_conn.execute(text("DROP TABLE IF EXISTS #check_transvar_p"))
check_transvar_p.to_sql("#check_transvar_p", work_conn, if_exists="replace", index=False)


In [ ]:
# DELETE FROM CHECK_TRANSVAR_P WHERE C_PARTIDA='' AND C_CAPITULO='' AND C_PROGRAMA='' AND C_ANALITICO=''
res = work_conn.execute(text("""
DELETE FROM #check_transvar_p
WHERE C_PARTIDA = '' AND C_CAPITULO = '' AND C_PROGRAMA = '' AND C_ANALITICO = ''
"""))
_log("DELETE #check_transvar_p (filas vacías)", res.rowcount)


In [ ]:
# ALTER TABLE CHECK_TRANSVAR_P MODIFY C_ACTIVO CHAR(6)
work_conn.execute(text("ALTER TABLE #check_transvar_p ALTER COLUMN C_ACTIVO CHAR(6)"))


In [ ]:
# UPDATE OBS='' WHERE OBS='Ingresar texto si aplica'
res = work_conn.execute(text("UPDATE #check_transvar_p SET OBS = '' WHERE OBS = :obs_texto"), {"obs_texto": "Ingresar texto si aplica"})
_log("UPDATE #check_transvar_p (limpia OBS placeholder)", res.rowcount)


In [ ]:
# UPDATE OBS='' WHERE C_SCN NOT IN (...) AND OBS NOT IN ('')
res = work_conn.execute(text("""
UPDATE #check_transvar_p
SET OBS = ''
WHERE C_SCN NOT IN ('D63-P2', 'D63-P', 'P3', 'P2') AND OBS NOT IN ('')
"""))
_log("UPDATE #check_transvar_p (limpia OBS fuera de C_SCN)", res.rowcount)


In [ ]:
# UPDATE OBS='CR18_difcoy', C_ACTIVO='D63' WHERE C_SCN IN ('D63-P2','D63-P') AND OBS=''
res = work_conn.execute(text("""
UPDATE #check_transvar_p
SET OBS = 'CR18_difcoy', C_ACTIVO = 'D63'
WHERE C_SCN IN ('D63-P2', 'D63-P') AND OBS = ''
"""))
_log("UPDATE #check_transvar_p (marca CR18_difcoy)", res.rowcount)


In [ ]:
# UPDATE OBS='CR18_difact' WHERE C_SCN='P3' AND OBS=''
res = work_conn.execute(text("UPDATE #check_transvar_p SET OBS = 'CR18_difact' WHERE C_SCN = 'P3' AND OBS = ''"))
_log("UPDATE #check_transvar_p (marca CR18_difact P3)", res.rowcount)


In [ ]:
# UPDATE OBS='CR18_difact' WHERE C_SCN='P2' AND C_CUP IN ('EEPP','SSPP') AND OBS=''
res = work_conn.execute(text("""
UPDATE #check_transvar_p
SET OBS = 'CR18_difact'
WHERE C_SCN = 'P2' AND C_CUP IN ('EEPP', 'SSPP') AND OBS = ''
"""))
_log("UPDATE #check_transvar_p (marca CR18_difact P2)", res.rowcount)


In [ ]:
# ANEXA CLASIFICACIONES FALTANTES
# Elimina de la propuesta las combinaciones que YA existen en la tabla maestra (anti-join)
res = work_conn.execute(text("""
DELETE FROM #check_transvar_p
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.CR18_T_SCN_2 t2
    WHERE t2.C_PARTIDA = #check_transvar_p.C_PARTIDA
      AND t2.C_CAPITULO = #check_transvar_p.C_CAPITULO
      AND t2.C_PROGRAMA = #check_transvar_p.C_PROGRAMA
      AND t2.T_CUENTA = #check_transvar_p.T_CUENTA
      AND t2.C_CUENTA = #check_transvar_p.C_CUENTA
      AND t2.C_ITEM = #check_transvar_p.C_ITEM
      AND t2.C_ASIGNACION = #check_transvar_p.C_ASIGNACION
      AND t2.C_ANALITICO = #check_transvar_p.C_ANALITICO
      AND t2.ENTIDAD = #check_transvar_p.ENTIDAD
)
"""))
_log("DELETE #check_transvar_p (ya existentes en CR18_T_SCN_2)", res.rowcount)


In [ ]:
# PROC DATASETS APPEND BASE=GOBGENER.CR18_T_SCN_2 DATA=CHECK_TRANSVAR_P FORCE
# FORCE de PROC APPEND: alinea por nombre de columna, columnas explícitas
cols_cr18_t_scn_2 = "ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, C_SCN, C_ACTIVO, C_CUP, OBS"
sql_append_cr18 = f"""
INSERT INTO GOBGENER.dbo.CR18_T_SCN_2 ({cols_cr18_t_scn_2})
SELECT {cols_cr18_t_scn_2}
FROM #check_transvar_p
"""
res = work_conn.execute(text(sql_append_cr18))
_log("APPEND GOBGENER.dbo.CR18_T_SCN_2", res.rowcount)


In [ ]:
# Trae check_transvar_p final a memoria (dataset WORK de salida del nodo)
check_transvar_p = pd.read_sql(text("SELECT * FROM #check_transvar_p"), work_conn)
_log("check_transvar_p", check_transvar_p)


In [ ]:
# VERIFICA QUE NO EXISTAN DUPLICADOS EN LA BASE DE CLASIFICACIONES VARIABLES
sql_check_analit_var = """
SELECT ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, COUNT(C_SCN) AS ID
FROM GOBGENER.dbo.CR18_T_SCN_2
GROUP BY ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, T_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO
"""
check_analit_var = pd.read_sql(text(sql_check_analit_var), engine)
# SI HAY REPETIDOS MOSTRARÁ CAMPOS EN ESTA SALIDA
check_analit_var = check_analit_var[check_analit_var["ID"] != 1]
_log("check_analit_var", check_analit_var)


## OPC 2a-Check Nvas TCons_MOD

Identifica cuentas 24/33 sin clasificación de Transferencias Condicionadas (TCons) vigente, descartando ítem 08 sin código SCN 'TC' y marcando como 'OK' los que no aplican, para revisión manual en Excel

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE con múltiples LEFT JOIN + DELETE/UPDATE de control + PROC EXPORT a Excel*

In [ ]:
# ========= OPC 2a-Check Nvas TCons_MOD =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
# Ejecucion dipres del año
# año interpolado como entero (no :param) para que la #tmp sobreviva
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2 ON (T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA)
WHERE T1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_dip))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #check_tcons_d"))
# CHECK COMPARADO CON ACTUAL MATRIZ DE ANALITICOS CLASIFICADOS COMO TCons
# año y cuentas/items interpolados como literales controlados para que la #tmp sobreviva
sql_check_tcons_d = f"""
SELECT DISTINCT t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO AS ANALITICO,
       t1.C_ENTIDAD AS ENTIDAD,
       t2.Clasif_Tcons,
       t2.M_Muestra,
       t3.C_SCN,
       t3.C_CAGENTE,
       t1.C_PARTIDA,
       t4.N_PARTIDA,
       t1.C_CAPITULO,
       t4.N_CAPITULO,
       t1.C_PROGRAMA,
       t4.N_PROGRAMA,
       t3.FECHA,
       t3.ANALISTA
INTO #check_tcons_d
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.ANALIT_TCONS t2 ON (t1.C_ENTIDAD = t2.ENTIDAD) AND (t2.C_ASIGNACION = t1.C_ASIGNACION)
    AND (t2.C_ANALITICO = t1.C_ANALITICO) AND (t2.C_CUENTA = t1.C_CUENTA) AND (t2.C_ITEM = t1.C_ITEM)
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t3 ON (t1.C_PARTIDA = t3.C_PARTIDA) AND (t1.C_CAPITULO = t3.C_CAPITULO)
    AND (t1.C_PROGRAMA = t3.C_PROGRAMA) AND (t1.C_ASIGNACION = t3.C_ASIGNACION)
    AND (t1.C_ANALITICO = t3.C_ANALITICO) AND (t1.C_CUENTA = t3.C_CUENTA) AND (t1.C_ITEM = t3.C_ITEM) AND (t1.C_ENTIDAD = t3.Entidad)
LEFT JOIN GOBGENER.dbo.ENTIDADES t4 ON (T1.C_ENTIDAD = T4.C_ENTIDAD AND t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA)
WHERE t2.Clasif_Tcons IS NULL AND t1.AÑO = {int(Year)} AND t1.C_CUENTA IN ('24','33') AND t1.C_ITEM IN ('01','03','06','09','08')
"""
work_conn.execute(text(sql_check_tcons_d))


In [ ]:
res = work_conn.execute(text("DELETE FROM #check_tcons_d WHERE C_ITEM = '08' AND C_SCN <> 'TC'"))
_log("DELETE #check_tcons_d (C_ITEM=08 y C_SCN<>TC)", res.rowcount)


In [ ]:
res = work_conn.execute(text("UPDATE #check_tcons_d SET Clasif_Tcons = 'OK' WHERE C_SCN NOT IN ('TC')"))
_log("UPDATE #check_tcons_d Clasif_Tcons=OK", res.rowcount)
check_tcons_d = pd.read_sql(text("SELECT * FROM #check_tcons_d"), work_conn)
_log("check_tcons_d", check_tcons_d)


In [ ]:
# PROC EXPORT DBMS=EXCELCS (bridge PC Files Server): se lee/escribe directo por la ruta del DATAFILE, sin servidor intermedio ni credenciales
ruta_check_tcons_d = Path("salidas") / "CHECK_TCONS_D.xlsb"
check_tcons_d.to_excel(ruta_check_tcons_d, sheet_name="TCONS", index=False)


In [ ]:
ejec_dip = pd.read_sql(text("SELECT * FROM #ejec_dip"), work_conn)
_log("ejec_dip", ejec_dip)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))


## OPC 2b-Clasif Nvas TCons_MOD

Importa clasificaciones TCONS revisadas desde Excel, normaliza y valida consistencia de llenado, anexa las nuevas clasificaciones a la tabla maestra y reporta cuentas TC aún sin clasificar

*confianza: medium · verificador: approve · SAS: PROC IMPORT EXCELCS + PROC SQL UPDATE/DELETE/CREATE + PROC DATASETS APPEND*

In [ ]:
# ========= OPC 2b-Clasif Nvas TCons_MOD =========
# IMPORTA CLASIFICACIONES
# PROC IMPORT DBMS=EXCELCS es el bridge PC Files Server de SAS: en Python se lee directo por su ruta,
# sin servidor intermedio ni credenciales de SERVER_NAME/SERVERUSER/SERVERPASS.
# Ruta relativa al workspace (la UNC original no es portable ni válida fuera del entorno SAS).
ruta_check_tcons_d = Path("datos") / "GobiernoGeneral" / "CHECK_TCONS_D.xlsb"
check_tcons_p = pd.read_excel(ruta_check_tcons_d, sheet_name="TCONS", engine="pyxlsb")
_log("check_tcons_p", check_tcons_p)


In [ ]:
check_tcons_p.to_sql("#check_tcons_p", work_conn, if_exists="replace", index=False)


In [ ]:
work_conn.execute(text("UPDATE #check_tcons_p SET CLASIF_TCONS = UPPER(CLASIF_TCONS)"))
work_conn.execute(text("UPDATE #check_tcons_p SET M_MUESTRA = 0 WHERE CLASIF_TCONS IN ('M','MM','MGCG','MGG')"))


In [ ]:
# DEFINE REGISTROS A CARGAR CON PROBLEMAS DE CONSISTENCIA EN SU LLENADO
work_conn.execute(text("DROP TABLE IF EXISTS #check_tcons_r"))
sql_check_tcons_r = """
SELECT *
INTO #check_tcons_r
FROM #check_tcons_p
WHERE CLASIF_TCONS NOT IN ('M','MM','MGCG','MGG','OK')
  AND (C_CUENTA IS NOT NULL AND C_CUENTA <> '')
"""
work_conn.execute(text(sql_check_tcons_r))
check_tcons_r = pd.read_sql(text("SELECT * FROM #check_tcons_r"), work_conn)
_log("check_tcons_r", check_tcons_r)


In [ ]:
res = work_conn.execute(text("DELETE FROM #check_tcons_p WHERE ENTIDAD IS NULL"))
_log("DELETE #check_tcons_p ENTIDAD faltante", res.rowcount)
res = work_conn.execute(text("DELETE FROM #check_tcons_p WHERE CLASIF_TCONS IS NULL OR CLASIF_TCONS = ''"))
_log("DELETE #check_tcons_p CLASIF_TCONS faltante", res.rowcount)


In [ ]:
# ANEXA CLASIFICACIONES FALTANTES
res = work_conn.execute(text("""
DELETE FROM #check_tcons_p
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.ANALIT_TCONS a
    WHERE a.C_CUENTA = #check_tcons_p.C_CUENTA
      AND a.C_ITEM = #check_tcons_p.C_ITEM
      AND a.C_ASIGNACION = #check_tcons_p.C_ASIGNACION
      AND a.C_ANALITICO = #check_tcons_p.C_ANALITICO
      AND a.ENTIDAD = #check_tcons_p.ENTIDAD
)
"""))
_log("DELETE #check_tcons_p ya existentes en ANALIT_TCONS", res.rowcount)


In [ ]:
# APPEND server-side: el FORCE de PROC APPEND alinea por nombre de columna
cols_analit_tcons = ", ".join(check_tcons_p.columns)
sql_append_analit_tcons = f"""
INSERT INTO GOBGENER.dbo.ANALIT_TCONS ({cols_analit_tcons})
SELECT {cols_analit_tcons}
FROM #check_tcons_p
"""
res = work_conn.execute(text(sql_append_analit_tcons))
_log("APPEND GOBGENER.dbo.ANALIT_TCONS", res.rowcount)
check_tcons_p = pd.read_sql(text("SELECT * FROM #check_tcons_p"), work_conn)
_log("check_tcons_p", check_tcons_p)


In [ ]:
# Verifica consistencia tcons en tabla 2 de clasificaciones: FALTANTES EN TABLA ANALIT_TCONS
sql_check_tcons_falt = """
SELECT T1.ENTIDAD, T1.C_CUENTA, T1.C_ITEM, T1.C_ANALITICO, T1.ANALITICO, T1.C_SCN,
       T1.C_CAGENTE, T1.FECHA, T1.ANALISTA, T2.CLASIF_TCONS
FROM GOBGENER.dbo.CR18_T_SCN_2 T1
LEFT JOIN GOBGENER.dbo.ANALIT_TCONS T2
    ON T1.ENTIDAD = T2.ENTIDAD AND T1.C_CUENTA = T2.C_CUENTA
   AND T1.C_ITEM = T2.C_ITEM AND T1.C_ANALITICO = T2.C_ANALITICO
WHERE T1.C_SCN = 'TC' AND (T2.CLASIF_TCONS IS NULL OR T2.CLASIF_TCONS = '')
ORDER BY T1.FECHA
"""
check_tcons_falt = pd.read_sql(text(sql_check_tcons_falt), engine)
_log("check_tcons_falt", check_tcons_falt)


## OPC 3a-EstTCONS_M_MM

Asigna a las TCONS clasificadas como mixtas (M) o múltiples (MM) sin muestra la estructura de desglose mensual (ponderaciones) existente para la combinación placeholder, y acumula lo nuevo en las tablas de estructuras correspondientes evitando duplicados

*confianza: medium · verificador: approve · SAS: PROC SQL: CREATE TABLE con claves placeholder, LEFT JOIN, DELETE con condición NULL, anti-join DELETE + PROC APPEND FORCE*

In [ ]:
# ========= OPC 3a-EstTCONS_M_MM =========
# DEFINE APERTURA DE TCONS A UTILIZAR
# Clasificar en esta tabla en campos C_ANAL y ENT la estructura que se quiere asignar a lo faltante, basado en las estructuras existentes
work_conn.execute(text("DROP TABLE IF EXISTS #tcons_est_def_m"))
sql_tcons_est_def_m = """
SELECT C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, ANALITICO, ENTIDAD, Clasif_Tcons, C_CAGENTE,
       '00000000' AS C_ANAL, 0 AS ENT
INTO #tcons_est_def_m
FROM #check_tcons_p
WHERE Clasif_Tcons IN ('M', 'MM') AND M_Muestra = 0
"""
work_conn.execute(text(sql_tcons_est_def_m))
tcons_est_def_m = pd.read_sql(text("SELECT * FROM #tcons_est_def_m"), work_conn)
_log("tcons_est_def_m", tcons_est_def_m)


In [ ]:
# busca estructuras de las M
work_conn.execute(text("DROP TABLE IF EXISTS #est_m"))
sql_est_m = """
SELECT T1.C_CUENTA, T1.C_ITEM, T1.ENTIDAD, T1.C_ASIGNACION, T1.C_ANALITICO,
       T2.MES, T2.C_SCN, T2.POND, T2.C_CAGENTE, T2.C_CUP
INTO #est_m
FROM #tcons_est_def_m T1
LEFT JOIN GOBGENER.dbo.EST_EXP_MIXTAS_M T2
    ON T1.C_CUENTA = T2.C_CUENTA AND T1.C_ANAL = T2.C_ANALITICO AND T1.ENT = T2.ENTIDAD
WHERE T1.Clasif_Tcons = 'M'
"""
work_conn.execute(text(sql_est_m))
# el LEFT JOIN sin match deja ENTIDAD en NULL (equivalente al ENTIDAD=. de SAS)
res = work_conn.execute(text("DELETE FROM #est_m WHERE ENTIDAD IS NULL"))
_log("DELETE #est_m sin match", res.rowcount)


In [ ]:
# busca estructuras de las MM
work_conn.execute(text("DROP TABLE IF EXISTS #est_mm"))
sql_est_mm = """
SELECT T1.C_CUENTA, T1.C_ITEM, T1.ENTIDAD, T1.C_ASIGNACION, T1.C_ANALITICO,
       T2.MES, T2.C_SCN, T2.POND, T2.C_CAGENTE
INTO #est_mm
FROM #tcons_est_def_m T1
LEFT JOIN GOBGENER.dbo.EST_EXP_MM T2
    ON T1.C_CUENTA = T2.C_CUENTA AND T1.C_ANAL = T2.C_ANALITICO AND T1.ENT = T2.ENTIDAD
WHERE T1.Clasif_Tcons = 'MM'
"""
work_conn.execute(text(sql_est_mm))
# el LEFT JOIN sin match deja ENTIDAD en NULL (equivalente al ENTIDAD=. de SAS)
res = work_conn.execute(text("DELETE FROM #est_mm WHERE ENTIDAD IS NULL"))
_log("DELETE #est_mm sin match", res.rowcount)


In [ ]:
# Anexa datos de estructuras tcons: descarta de EST_M lo que ya existe en la tabla destino (anti-join)
res = work_conn.execute(text("""
DELETE FROM #est_m
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.EST_EXP_MIXTAS_M b
    WHERE b.C_CUENTA = #est_m.C_CUENTA AND b.C_ITEM = #est_m.C_ITEM
      AND b.C_ASIGNACION = #est_m.C_ASIGNACION AND b.C_ANALITICO = #est_m.C_ANALITICO
      AND b.ENTIDAD = #est_m.ENTIDAD
)
"""))
_log("DELETE #est_m ya existentes en EST_EXP_MIXTAS_M", res.rowcount)

# PROC APPEND FORCE: alinea por nombre, server-side desde la #tmp
cols_est_m = "C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO, MES, C_SCN, POND, C_CAGENTE, C_CUP"
sql_append_est_m = f"""
INSERT INTO GOBGENER.dbo.EST_EXP_MIXTAS_M ({cols_est_m})
SELECT {cols_est_m}
FROM #est_m
"""
res = work_conn.execute(text(sql_append_est_m))
_log("APPEND GOBGENER.dbo.EST_EXP_MIXTAS_M", res.rowcount)
est_m = pd.read_sql(text("SELECT * FROM #est_m"), work_conn)
_log("est_m", est_m)


In [ ]:
# Anexa datos de estructuras tcons (MM): descarta de EST_MM lo que ya existe en la tabla destino (anti-join)
res = work_conn.execute(text("""
DELETE FROM #est_mm
WHERE EXISTS (
    SELECT 1 FROM GOBGENER.dbo.EST_EXP_MM b
    WHERE b.C_CUENTA = #est_mm.C_CUENTA AND b.C_ITEM = #est_mm.C_ITEM
      AND b.C_ASIGNACION = #est_mm.C_ASIGNACION AND b.C_ANALITICO = #est_mm.C_ANALITICO
      AND b.ENTIDAD = #est_mm.ENTIDAD
)
"""))
_log("DELETE #est_mm ya existentes en EST_EXP_MM", res.rowcount)

# PROC APPEND FORCE: alinea por nombre, server-side desde la #tmp
cols_est_mm = "C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO, MES, C_SCN, POND, C_CAGENTE"
sql_append_est_mm = f"""
INSERT INTO GOBGENER.dbo.EST_EXP_MM ({cols_est_mm})
SELECT {cols_est_mm}
FROM #est_mm
"""
res = work_conn.execute(text(sql_append_est_mm))
_log("APPEND GOBGENER.dbo.EST_EXP_MM", res.rowcount)
est_mm = pd.read_sql(text("SELECT * FROM #est_mm"), work_conn)
_log("est_mm", est_mm)


In [ ]:
# PROC SQL DROP TABLE EST_M, EST_MM (limpieza de temporales de sesión)
for t in ["#est_m", "#est_mm"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## OPC 3b-EstTCONS_CHECK

Calcula el total de ponderación mensual por cuenta-ítem-entidad-asignación-analítico para las estructuras mixtas y fijas, y deja solo los casos donde el total NO cuadra en 1 (100%)

*confianza: high · verificador: approve · SAS: PROC SQL CREATE TABLE con GROUP BY + SUM/ROUND, y DELETE de filas consistentes*

In [ ]:
# ========= OPC 3b-EstTCONS_CHECK =========
work_conn.execute(text("DROP TABLE IF EXISTS #check_mixtas_tot"))
# CHEQUEA QUE SUMEN 1 LAS ESTRUCTURAS
sql_check_mixtas_tot = """
SELECT MES, C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO,
       SUM(POND) AS TOTAL,
       ROUND(SUM(POND), 1) AS TOTAL2
INTO #check_mixtas_tot
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M
GROUP BY MES, C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO
"""
work_conn.execute(text(sql_check_mixtas_tot))


In [ ]:
res = work_conn.execute(text("DELETE FROM #check_mixtas_tot WHERE ROUND(TOTAL2, 0) = 1"))
_log("DELETE #check_mixtas_tot", res.rowcount)
check_mixtas_tot = pd.read_sql(text("SELECT * FROM #check_mixtas_tot"), work_conn)
_log("check_mixtas_tot", check_mixtas_tot)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #check_m_tot"))
sql_check_m_tot = """
SELECT MES, C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO,
       SUM(POND) AS TOTAL,
       ROUND(SUM(POND), 1) AS TOTAL2
INTO #check_m_tot
FROM GOBGENER.dbo.EST_EXP_MM
GROUP BY MES, C_CUENTA, C_ITEM, ENTIDAD, C_ASIGNACION, C_ANALITICO
"""
work_conn.execute(text(sql_check_m_tot))


In [ ]:
res = work_conn.execute(text("DELETE FROM #check_m_tot WHERE ROUND(TOTAL2, 0) = 1"))
_log("DELETE #check_m_tot", res.rowcount)
check_m_tot = pd.read_sql(text("SELECT * FROM #check_m_tot"), work_conn)
_log("check_m_tot", check_m_tot)
